# Step 2 · 3주차 실습 — 모델 비교와 통계 검정
SVM · RandomForest · XGBoost를 **같은 데이터·같은 분할**로 학습하고, hold-out → 혼동행렬 → 5-fold → 대응표본 t-검정 → 계산 비용 순으로 비교한다.

**실습 전 유의사항 (용어 정리 4~7쪽)**
- 세 모델은 데이터를 나누는 방식이 근본적으로 다르다 (SVM=최대 마진+커널, RF=트리 다수결, XGB=오차 보정 부스팅). SVM은 표준화 필수 → `dataset.npz`는 이미 학습셋 기준으로 표준화됨.
- 비교는 **완전히 대칭적이지 않다**: SVM·RF는 기본값, XGBoost만 지정 설정(400개/depth 6/lr 0.05). `random_state=42` 고정.
- **Macro 평균**은 실습에서 고른 방식이다 (리포트에 명시). 평균만 쓰면 절반만 보고한 것 → **표준편차 병기**.
- 이 결과는 **trial-wise 분할** 성능이다. 처음 보는 사람(LOSO)에 대한 성능이 아니다 → 한계 항목에 반드시 기재.
- p ≥ 0.05는 *차이 미검출*이지 *동등성의 증거*가 아니다. 비모수 검정이 작은 표본에서 항상 더 안전한 것은 아니다.
- 높은 정확도·NaN만으로 누수를 단정하지 않는다. 분할·검정 조건을 먼저 확인한다.

## 0. 설정 — 2주차 파이프라인 재사용

In [ ]:
import os, glob, time, subprocess, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.signal import butter, iirnotch, filtfilt
from scipy.stats import ttest_rel, wilcoxon
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.base import clone                      # import 누락 주의
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier                   # pip install xgboost
warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False
# Colab 한글 폰트: !apt-get -qq install fonts-nanum  후  plt.rcParams['font.family']='NanumGothic'

REPO = 'https://github.com/TestUser-alt-dev/sEMG-Hand-Gesture-Elbow-Angle-Dataset.git'
REPO_DIR = 'sEMG-Hand-Gesture-Elbow-Angle-Dataset'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','--depth','1',REPO,REPO_DIR], check=True)
DATA_DIR = os.path.join(REPO_DIR,'data'); FS = 1000

# 2주차 dataset.npz (없으면 내 저장소에서 받기)
if not os.path.exists('dataset.npz'):
    subprocess.run(['wget','-q','https://github.com/starting-stud/sEMG_1012/raw/main/dataset.npz'], check=True)

In [ ]:
def load_trials():
    out = []
    for f in sorted(glob.glob(os.path.join(DATA_DIR,'*.csv'))):
        b = os.path.basename(f)[:-4]; subj, ang, g = b[0], int(b[2:4]), b[5:]
        d = pd.read_csv(f); x = d.iloc[:,:5].values.astype(float)
        for t, s in enumerate(np.where(d['mark'].values != 0)[0]):
            out.append(dict(x=x[s:s+5000], subj=subj, angle=ang, gesture=g, trial=t, file=b))
    return out
def preprocess(x, fs=FS):
    bn, an = iirnotch(60, 30, fs); x = filtfilt(bn, an, x, axis=0)
    b, a = butter(4, [20/(fs/2), 450/(fs/2)], btype='band'); return filtfilt(b, a, x, axis=0)
def make_windows(x, win=200, hop=100):
    n = (len(x)-win)//hop + 1; return np.stack([x[i*hop:i*hop+win] for i in range(n)])
def td_features(w, th=0.01):
    rms = np.sqrt(np.mean(w**2,axis=0)); mav = np.mean(np.abs(w),axis=0)
    d = np.diff(w,axis=0); wl = np.sum(np.abs(d),axis=0)
    zc = np.sum(((w[:-1]*w[1:])<0)&(np.abs(d)>=th),axis=0); ssc = np.sum((d[:-1]*d[1:])<0,axis=0)
    return np.concatenate([rms,mav,wl,zc,ssc])

T = load_trials()
cls_names = [f'{g}_{a}' for a in (30,60) for g in ['fist','open','pro','extor']]
labels = np.array([cls_names.index(f"{t['gesture']}_{t['angle']}") for t in T])
idx = np.arange(len(T))
# 시행별 특징을 표준화 전 상태로 한 번만 계산 -> 교차검증에서는 인덱스로 나눈다 (속도)
FEAT = np.stack([np.stack([td_features(w) for w in make_windows(preprocess(t['x'])[1000:3000])]) for t in T])
print('시행별 특징 배열:', FEAT.shape)   # (1920, 19, 25)

## 단계 1. 세 모델 학습 — 조건이 하나라도 다르면 비교가 성립하지 않는다

In [ ]:
d = np.load('dataset.npz', allow_pickle=True)
Xtr, ytr, Xte, yte = d['Xtr'], d['ytr'], d['Xte'], d['yte']
le = LabelEncoder(); ytr = le.fit_transform(ytr); yte = le.transform(yte)   # XGBoost는 0부터 시작하는 정수 라벨
print('classes_:', le.classes_, '| 이름:', list(d['class_names']))

# 분할 일관성 확인: 재구성한 시행 분할이 dataset.npz와 같은가
tr_i, te_i = train_test_split(idx, test_size=0.2, stratify=labels, random_state=42)
assert set(np.unique(d['trial_tr'])) == set(tr_i) and set(np.unique(d['trial_te'])) == set(te_i)

models = {
 'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', decision_function_shape='ovr'),
 'RF' : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
 'XGB': XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                      subsample=0.9, colsample_bytree=0.8, random_state=42),
}
for k, m in models.items():
    m.fit(Xtr, ytr); print(k, '학습 완료')

## 단계 2. hold-out 평가 — 정확도만 보지 않는다
※ precision/recall/F1은 **macro 평균** (실습 선택). 리포트에 명시할 것.

In [ ]:
rows = []
for k, m in models.items():
    p = m.predict(Xte)
    rows.append([k, accuracy_score(yte,p), precision_score(yte,p,average='macro'),
                 recall_score(yte,p,average='macro'), f1_score(yte,p,average='macro')])
df = pd.DataFrame(rows, columns=['model','acc','precision','recall','f1']); print(df.round(4))

## 단계 3. 혼동행렬 — 라벨을 직접 대조하고, 인접 칸이라는 이유만으로 오류 종류를 단정하지 않는다

In [ ]:
L = list(d['class_names'])          # le.classes_ 는 0~7 정수이므로 이름은 class_names로 대조
cm = confusion_matrix(yte, models['RF'].predict(Xte))
fig, ax = plt.subplots(figsize=(6.5,5.5)); im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(8), L, rotation=45); ax.set_yticks(range(8), L)
ax.set_xlabel('예측'); ax.set_ylabel('실제')
for i in range(8):
    for j in range(8):
        if cm[i,j]: ax.text(j,i,cm[i,j],ha='center',va='center',fontsize=8,
                            color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.savefig('confusion8.png', dpi=120); plt.show()

# 오류를 라벨 기준으로 분해: 같은 동작·다른 각도 / 다른 동작·같은 각도 / 둘 다 다름
g = lambda s: s.split('_')[0]; a = lambda s: s.split('_')[1]
cnt = dict(same_gesture_diff_angle=0, diff_gesture_same_angle=0, diff_both=0)
for i in range(8):
    for j in range(8):
        if i == j: continue
        key = ('same_gesture_diff_angle' if g(L[i])==g(L[j]) else
               'diff_gesture_same_angle' if a(L[i])==a(L[j]) else 'diff_both')
        cnt[key] += cm[i,j]
print(cnt, '| 칸 수: 4 / 24 / 24  -> 칸당 평균', {k: round(v/n,1) for (k,v),n in zip(cnt.items(),[8,24,24])})

## 단계 4. 5-fold 교차검증 — **시행 단위**로 나누고, scaler는 각 학습 폴드에서만 fit
평균과 표준편차를 함께 구한다. (윈도우 단위로 나누면 누수.)

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=42)
fold_acc = {k: [] for k in models}
for f, (a_i, b_i) in enumerate(skf.split(idx, labels)):
    assert not (set(a_i) & set(b_i))
    Xa = FEAT[a_i].reshape(-1,25); ya = np.repeat(labels[a_i], 19)
    Xb = FEAT[b_i].reshape(-1,25); yb = np.repeat(labels[b_i], 19)
    sc = StandardScaler(); Xa = sc.fit_transform(Xa); Xb = sc.transform(Xb)
    ya, yb = le.transform(ya), le.transform(yb)
    for k, proto in models.items():
        m = clone(proto); m.fit(Xa, ya); fold_acc[k].append(m.score(Xb, yb))
    print('fold', f+1, '완료', flush=True)
for k, v in fold_acc.items():
    print(f'{k}: {np.mean(v)*100:.2f} ± {np.std(v)*100:.2f} %   (폴드별: {[round(x*100,2) for x in v]})')

## 단계 5. 통계 검정 — 폴드별 정확도로 대응표본 t-검정
**해석 주의**: ① p ≥ 0.05는 동등성의 증거가 아니다. ② 5개 폴드는 학습 데이터가 서로 겹쳐 독립이 아니므로 p값은 **낙관적**일 수 있다(표본 수 5). ③ 비모수(Wilcoxon)는 n=5에서 **p가 0.0625 아래로 내려갈 수 없다** — 더 안전한 것이 아니라 검정력이 없는 것.

In [ ]:
for a_, b_ in combinations(models.keys(), 2):
    diff = np.array(fold_acc[a_]) - np.array(fold_acc[b_])
    t, p = ttest_rel(fold_acc[a_], fold_acc[b_])
    if not np.isfinite(p): print(a_, b_, '검정 불가'); continue
    verdict = '유의함' if p < 0.05 else '유의하지 않음'
    print(f'{a_} vs {b_}: t = {t:.3f}, p = {p:.4g}  ({verdict}) | 평균 차이 {diff.mean()*100:+.2f}%p, 차이 sd {diff.std(ddof=1)*100:.2f}%p, Wilcoxon p = {wilcoxon(fold_acc[a_],fold_acc[b_]).pvalue:.4f}')

## 단계 6. 계산 비용 — 측정 환경 차이를 기록하고 실제 배포 성능과 구분한다

In [ ]:
for k, m in models.items():
    m = clone(m); t0 = time.time(); m.fit(Xtr, ytr); train_t = time.time()-t0
    t0 = time.time()
    for _ in range(100): m.predict(Xte[:1])
    infer_ms = (time.time()-t0)*10
    print(f'{k}: 학습 {train_t:.3f}s, 추론 {infer_ms:.3f} ms/sample')
# 측정 환경(CPU/코어 수)을 리포트에 기록할 것 - 순위 위주로 해석